In [ ]:
# import kagglehub

In [ ]:
# download the dataset from kaggle

# kagglehub.dataset_download("undefinenull/million-song-dataset-spotify-lastfm")

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent
data_path = root / 'data/raw'
song_data_path = data_path / 'Music Info.csv'
user_data_path = root / 'data/local/listens.csv'

# Song Dataset

In [ ]:
# load the song data

df_songs = pd.read_csv(song_data_path)
df_songs.sample(3)

## Preliminary Analysis

In [ ]:
# size of the dataset

df_songs.shape

In [ ]:
# data indo

df_songs.info()

In [ ]:
df_songs['spotify_preview_url']

In [ ]:
# drop columns from data

columns_to_drop = ['spotify_preview_url']

df_songs.drop(columns=columns_to_drop,inplace=True)

df_songs.sample(3)

### Missing values

In [ ]:
# missing values

df_songs.isnull().sum()

In [ ]:
df_songs.isna().mean().sort_values().plot.barh(title='Missing values by field')

In [ ]:
# percent of missing values in the data

df_songs.isnull().sum()

In [ ]:
df_songs.isnull().mean().sort_values(ascending=False).mul(100).head(2)

### Duplicates in data

In [ ]:
df_songs.duplicated().sum()

In [ ]:
# duplicates based on name of the song

(
df_songs
.assign(name = df_songs['name'].str.lower())
.duplicated(subset='name')
.sum()
)

In [ ]:
# rows that as duplicate
(
df_songs
.loc[
    df_songs
    .assign(name = df_songs['name'].str.lower())
    .duplicated(subset='name',keep=False)
]
.assign(name = df_songs['name'].str.lower())
.sort_values("name")
.head(6)
)

In [ ]:
# duplicate based on spotify id

df_songs.duplicated(subset="spotify_id").sum()

In [ ]:
df_songs.duplicated(subset=["spotify_id","year","duration_ms"]).sum()

In [ ]:
df_songs[df_songs.duplicated(subset=["spotify_id","year","duration_ms"],keep=False)]

In [ ]:
# drop duplicates

df_songs.drop_duplicates(subset=["spotify_id","year","duration_ms"],inplace=True)

df_songs.shape

In [ ]:
df_songs.duplicated(subset=["spotify_id","year","duration_ms"]).sum()

**Duplicated songs has been removed from the data**

### Columnwise Analysis

#### Categorical Column

In [ ]:
# list of columns in the data

df_songs.columns

In [ ]:
df_songs.dtypes

In [ ]:
# categorical columns

categorical_columns = df_songs.select_dtypes(include="object").columns

categorical_columns

In [ ]:
def categorical_analysis(df,feature_names,k_artist=15):

    for feature in feature_names:

        print(f"The number of categories in {feature} are {df[feature].nunique()}")
        print('*'*75)

        if feature in ['artist','genre']:
            print(f"Top {k_artist} caregories in {feature} are :")
            print(df[feature].value_counts().head(k_artist))



In [ ]:
categorical_analysis(df_songs,categorical_columns)

**Observations:**

- The Track ID and Spotify ID's are unique for every row.
- The song names have some repititions. That is because there are a few songs in the data whose name is the same but is sung by a different artist.
- The dataset contains songs of about 8317 artists.
- There are 15 distinct categories in the Genre column.

In [ ]:
# countplot of genre

sns.countplot(x='genre',data=df_songs)
plt.xticks(rotation=45)
plt.show()

In [ ]:
genre_sample = df_songs.groupby('genre')[['genre','artist']].sample(2)

genre_sample

In [ ]:
df_songs[df_songs['name'].map(lambda value: isinstance(value, str) and not value.isascii())].head()

In [ ]:
df_songs[df_songs['artist'].map(lambda value: isinstance(value, str) and not value.isascii())].head()

In [ ]:
# tags

df_songs['tags'].value_counts()

In [ ]:
all_tags = []

for tags in df_songs['tags'].dropna().str.strip().str.lower().str.replace(" ","").str.split(','):
    all_tags.extend(tags)

all_tags

In [ ]:
len(set(all_tags))

### Integer based Columns

**Key**

What it means: The musical key in which the song is composed, represented as integers (e.g., 0 = C, 1 = C#, 2 = D, etc., up to 11).

- When value is small: Represents keys like C or D, often associated with simplicity or familiarity.

- When value is large: Represents keys like A# or B, which may sound brighter or more complex, depending on the context.

**Mode**

What it means: Refers to the modality of the song, where 0 = minor (often associated with sadness or tension) and 1 = major (associated with happiness or resolution).

- When value is small: Indicates the song is in a minor mode, creating a more somber or serious tone.

- When value is large: Indicates the song is in a major mode, giving it a cheerful and uplifting feel.

**Time Signature**

What it means: The number of beats in each bar of music, typically expressed as an integer (e.g., 4 for 4/4 time).

- When value is small: Indicates fewer beats per measure (e.g., 3), which can create a waltz-like or less conventional rhythm.

- When value is large: Indicates more beats per measure (e.g., 5 or 7), often leading to a more complex or experimental rhythmic structure.

A value of 4 in the time signature typically means the song is in 4/4 time, also known as "common time." This means there are 4 beats per measure, and the quarter note gets one beat.

Impact of a 4 Time Signature:

- Musical Feel: It creates a steady, balanced, and natural rhythm that is easy to follow.
- Prevalence: It is the most common time signature in Western music, used in many genres like pop, rock, classical, and jazz.
- Examples: Songs like "Billie Jean" by Michael Jackson or "Shape of You" by Ed Sheeran are in 4/4.

### Integer Columns

In [ ]:
integer_columns = df_songs.select_dtypes(include=['integer']).columns

integer_columns

In [ ]:
df_songs[integer_columns]

In [ ]:
df_songs[integer_columns].drop(columns=['duration_ms']).astype('object').describe()

In [ ]:
# number of song per year in data
bins=df_songs['year'].max()-df_songs['year'].min()
sns.histplot(df_songs,x='year',bins=bins,stat='count')

In [ ]:
# most songs from each year(top 5)

df_songs.groupby('year').count()['track_id'].sort_values(ascending=False).head(5).reset_index()

**Keys**

Musical notes are assigned key values from 0 to 11, representing all the chromatic notes in an octave. Here’s the mapping:

| **Key Value** | **Note**        |
|:-------------:|:---------------:|
| **0**         | **C**           |
| **1**         | **C# / Db**     |
| **2**         | **D**           |
| **3**         | **D# / Eb**     |
| **4**         | **E**           |
| **5**         | **F**           |
| **6**         | **F# / Gb**     |
| **7**         | **G**           |
| **8**         | **G# / Ab**     |
| **9**         | **A**           |
| **10**        | **A# / Bb**     |
| **11**        | **B**           |

**Notes Explained:**

-  (#): Raise the note by a semitone (e.g., C → C#).
 - Flats (b): Lower the note by a semitone (e.g., D → Db).

This cycle repeats across octaves!

## 🎵 English vs Hindi Notes Comparison

| **Key Value** | **English Note** | **Hindi Music Note (Swaras)** |
|:-------------:|:----------------:|:-----------------------------:|
| **0**         | **C**            | **Sa**                        |
| **1**         | **C# / Db**      | **Komal Re**                  |
| **2**         | **D**            | **Re**                        |
| **3**         | **D# / Eb**      | **Komal Ga**                  |
| **4**         | **E**            | **Ga**                        |
| **5**         | **F**            | **Ma**                        |
| **6**         | **F# / Gb**      | **Tivra Ma**                  |
| **7**         | **G**            | **Pa**                        |
| **8**         | **G# / Ab**      | **Komal Dha**                 |
| **9**         | **A**            | **Dha**                       |
| **10**        | **A# / Bb**      | **Komal Ni**                  |
| **11**        | **B**            | **Ni**                        |

---

### 📝 Notes
- **Komal** = *flat (lowered) note*  
- **Tivra** = *sharp (raised) note*  
- The cycle repeats with **Sa** at the next octave 🎶

If a song has a key value of 2, it means the song is composed in the D major or D minor key, depending on the mode (0 = minor, 1 = major).

**Interpretation:**x

- Key Value 2 (D): The tonic note, or "home base," of the song is D. All other notes and chords in the song revolve around this note.

In Western Music:
- D Major: Bright and uplifting.
- D Minor: Melancholic or introspective.

In Hindi Classical Music: This corresponds to the "Re" swara (if D is treated as the Sa of the scale).
The overall mood or emotion of the song will depend on the mode and how the scale is used melodically and harmonically.

In [ ]:
df_songs['key'].value_counts().sort_index()

In [ ]:
# unique values in the key column

df_songs['key'].unique()

In [ ]:
# percentage of songs wrt to key in the data

temp = df_songs['key'].value_counts(normalize=True).mul(100).sort_index()

sns.barplot(x=temp.index,y=temp.values)
plt.title("Percentage of Songs wrt to Key")
plt.ylabel('Percentage')
plt.show()

In [ ]:
# countplot for mode

sns.countplot(x='mode',data=df_songs)
plt.show()

In [ ]:
# unique values for time signature

df_songs['time_signature'].value_counts()

In [ ]:
df_songs['time_signature'].unique()

In [ ]:
temp = df_songs['time_signature'].value_counts(normalize=True).mul(100)
sns.barplot(x=temp.index,y=temp.values)
plt.show()

In [ ]:
# statistical summary of time duration

(
df_songs[['duration_ms']]
.assign(duration_min = df_songs['duration_ms'].div(1000).div(60))
.drop(columns=['duration_ms'])
.describe()
 )

In [ ]:
# time duration histogram

time_duration_mins = df_songs["duration_ms"].div(1000).div(60)

sns.histplot(time_duration_mins)
plt.xlabel("Time Duration (mins)")
plt.show()

In [ ]:
# time duration boxplot

sns.boxplot(time_duration_mins)
plt.ylabel("Time Duration (mins)")
plt.show()

In [ ]:
# song that is longer than 60 mins

df_songs[time_duration_mins>60]

## Continuous Columns

In [ ]:
continuous_columns = df_songs.select_dtypes(include='float')

continuous_columns

**Danceability**

**What it means:** Measures how suitable a track is for dancing, based on tempo, rhythm stability, beat strength, and overall regularity.

- When value is small: The song may feel less rhythmic or harder to dance to, like ballads or experimental music.

- When value is large: The song is highly danceable, with a strong beat and rhythmic consistency, ideal for clubs or parties.

**Energy**

**What it means:** Represents the intensity and activity of a song, considering factors like loudness, tempo, and instrumentation.

- When value is small: The song feels mellow, calm, or laid-back, such as acoustic or chill tracks.

- When value is large: The song feels loud, energetic, and lively, often found in rock or EDM genres.

**Loudness**

What it means: The overall volume of the song in decibels (dB), averaged over the track.

- When value is small: The song is quieter, suitable for soft or ambient styles.
- When value is large: The song is louder and more powerful, typically associated with dynamic or energetic tracks.

**Speechiness**

What it means: Measures the presence of spoken words in a track. A higher value indicates more speech-like content.

- When value is small: The song is more musical, with fewer spoken or rap-like elements.
- When value is large: The track has spoken word, podcast-style content, or heavy rap influence.

**Acousticness**

What it means: Reflects how acoustic (non-electronic) the track is.

- When value is small: The track is more electronic or synthetic in nature.
- When value is large: The track is acoustic, featuring instruments like guitar, piano, or strings.

**Instrumentalness**

What it means: Predicts the likelihood of a song having no vocals (purely instrumental).

- When value is small: The track likely has vocals or lyrics.
- When value is large: The track is primarily instrumental, like classical or ambient music.

**Liveness**

What it means: Measures the presence of a live audience in the recording.

- When value is small: The track sounds studio-produced without live ambiance.
- When value is large: The track feels live, with audience sounds or a concert vibe.

**Valence**

What it means: Indicates the positivity or happiness of a track's mood.

- When value is small: The song feels sad, somber, or emotionally heavy.
- When value is large: The song feels cheerful, uplifting, or happy.

In [ ]:
def numerical_analysis(df,columns):
    for column in columns:
        print(f"Numerical Column Analysis of {column}")
        print(f"Statistical summary of {column}")
        print(df[column].describe())

        fig = plt.figure(figsize=(12,4))

        # histogram plot
        plt.subplot(1,2,1)
        sns.histplot(df[column])
        plt.title(f"Histogram for {column}")

        # boxplot for column
        plt.subplot(1,2,2)
        sns.boxplot(df[column])
        plt.title(f"Boxplot for {column}")
        plt.show()

        print('*'*100)
    
    print('*'*120)

    print("Pairplot")
    sns.pairplot(columns)
    plt.show()

In [ ]:
numerical_analysis(df_songs,continuous_columns)

# Users & Song Data

In [ ]:
# load the dataset

df_users = pd.read_csv(user_data_path)

df_users.sample(3)

In [ ]:
# shape of the dataset

df_users.shape

In [ ]:
# dataset info

df_users.info()

In [ ]:
# null values

df_users.isnull().sum()

In [ ]:
# duplication in dataset

df_users.duplicated().sum()

In [ ]:
# duplication based on track id

df_users[df_users.duplicated(subset=['user_id','track_id'],keep=False)]

In [ ]:
# unique user_id in the data

df_users['user_id'].nunique()

In [ ]:
# unique songs in the data

df_users['track_id'].nunique()

In [ ]:
# top 10 most played songs in user data

top_10_songs = df_users['track_id'].value_counts().head(10)

top_10_songs

In [ ]:
# details of top 10 songs from df_songs dataset

top_10_songs_track_id = top_10_songs.index.to_list()

df_songs[df_songs['track_id'].isin(top_10_songs_track_id)]

In [ ]:
# most playcounts for songs

top_10_played_songs = (
    df_users
    .groupby('track_id')['playcount']
    .agg(sum)
    .sort_values(ascending=False)
    .head(10)
 )

top_10_played_songs.reset_index()

In [ ]:
# top 10 played song index_id
top_10_played_songs_index_id = top_10_played_songs.index.to_list()

top_10_played_songs_index_id

In [ ]:
# checking top 10 most played songs in df_songs dataset

df_songs[df_songs['track_id'].isin(top_10_played_songs_index_id)]

In [ ]:
pd.concat([top_10_songs,top_10_played_songs],axis=1)

In [ ]:
# most diverse users

most_diverse_users = (
df_users
.groupby('user_id')['track_id']
.agg('count')
.sort_values(ascending=False)
.head(10)
)

most_diverse_users

In [ ]:
# most playcounts for users

most_active_user = (
    df_users
    .groupby('user_id')['playcount']
    .agg(sum)
    .sort_values(ascending=False)
    .head(10)
    )

most_active_user

In [ ]:
pd.concat([most_diverse_users,most_active_user],axis=1)